# Destination Projection — Historical Backtest & Error Diagnostics

Compares the Destination-Adjusted Projection model's **actual production output** (re-scored point-in-time per historical season) against **actual realized** per-game stats for real historical transfers, then mines residuals for systematic bias patterns.

Full design: `docs/models/destination_projection_backtest_plan.md`. This notebook does the *interactive* half of that plan (§7b clustering, cohort review, plots) — the non-interactive residual computation/logging lives in `scripts/run_destination_backtest.py`, matching every other model's interactive-notebook-vs-`run_*.py`-script split in this repo.

**Not duplicated here:** `fit_role_usage_model` CV (see `scheme_fit_scorer.ipynb`'s precedent / `destination_projection.py`'s own `compute_cohort_validation`), the residual load/compute functions (imported from `portalpoint.modeling.destination_backtest`, not reimplemented).

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

sys.path.insert(0, str(Path.cwd().parents[1] / "src"))

from portalpoint.modeling import destination_backtest as db
from portalpoint.modeling.destination_projection import MODEL_VERSION
from portalpoint.modeling.io import get_sync_engine

pd.set_option("display.max_columns", 50)
engine = get_sync_engine()

## 1. Load population, actual outcomes, projected outcomes, residuals

Reuses `destination_backtest.py`'s pure functions directly — no logic redefined here.

In [3]:
# min_dest_season=2023, not 2022 — 2022 confirmed infeasible (real
# run_playing_time.py failure): barttorvik data starts at season 2021, so
# target_season=2022's Playing Time model can't get the >=2 prior seasons it
# hard-requires. See destination_projection_backtest_plan.md §13.
MIN_SEASON, MAX_SEASON = 2023, 2026

population_df = db.load_backtest_population(engine, MIN_SEASON, MAX_SEASON)
print(f"Backtest population: {len(population_df):,} matched historical transfers")
population_df["dest_season"].value_counts().sort_index()

Backtest population: 3,168 matched historical transfers


dest_season
2023    565
2024    710
2025    932
2026    961
Name: count, dtype: int64

In [4]:
# Readiness check — which seasons actually have destination-mode player_projections
# rows already? (see scripts/run_destination_backtest.py --backfill for filling gaps)
from sqlalchemy import text

seasons = sorted(population_df["dest_season"].unique().tolist())
with engine.connect() as conn:
    present = {
        int(r[0])
        for r in conn.execute(
            text(
                "SELECT DISTINCT season FROM player_projections "
                "WHERE projection_mode = 'destination' AND model_version = :mv "
                "AND season = ANY(:seasons)"
            ),
            {"mv": MODEL_VERSION, "seasons": seasons},
        ).fetchall()
    }
missing = sorted(s for s in seasons if s not in present)
print(f"Seasons ready: {sorted(present)}")
print(f"Seasons missing (need backfill first): {missing}")

Seasons ready: [2023, 2024, 2025, 2026]
Seasons missing (need backfill first): []


In [5]:
actual_df = db.load_actual_outcomes(engine, population_df)
projected_df = db.load_projected_outcomes(engine, population_df, MODEL_VERSION)
residual_df = db.compute_residuals(actual_df, projected_df)

print(f"Actual outcomes (games-played floor applied): {len(actual_df):,}")
print(f"Projected outcomes found: {len(projected_df):,}")
print(f"Backtest rows with both (real n for this analysis): {len(residual_df):,}")
residual_df.head()

Actual outcomes (games-played floor applied): 2,895
Projected outcomes found: 3,043
Backtest rows with both (real n for this analysis): 2,782


,player_id,dest_school_id,dest_season,games_played,points_per_game,rebounds_per_game,assists_per_game,steals_per_game,blocks_per_game,turnovers_per_game,projected_box_score,explanation,proj_pts,proj_reb,proj_ast,proj_stl,proj_blk,proj_tov,residual_pts,pct_error_pts,residual_reb,pct_error_reb,residual_ast,pct_error_ast,residual_stl,pct_error_stl,residual_blk,pct_error_blk,residual_tov,pct_error_tov
0,3286031872075688894,45,2025,33,12.3030,7.4848,5.8788,1.6061,0.9091,1.8182,"{'ast_per_game': 1.85, 'blk_per_game': 0.49, '...","{'dest_tier': 1, 'gap_match': 55.73, 'scheme_f...",10.02,5.18,1.85,0.85,0.49,1.56,2.2830,18.56,2.3048,30.79,4.0288,68.53,0.7561,47.08,0.4191,46.10,0.2582,14.20
1,30574412514908185,196,2025,7,1.6667,1.5556,1.3333,0.2222,0.0000,1.5556,"{'ast_per_game': 1.93, 'blk_per_game': 0.23, '...","{'dest_tier': 1, 'gap_match': 30.43, 'scheme_f...",10.09,2.04,1.93,0.90,0.23,1.62,-8.4233,-505.39,-0.4844,-31.14,-0.5967,-44.75,-0.6778,-305.04,-0.2300,NaN,-0.0644,-4.14
2,5165044871727590183,102,2025,33,12.1818,5.3939,3.7576,1.6364,0.4242,0.7879,"{'ast_per_game': 1.51, 'blk_per_game': 0.45, '...","{'dest_tier': 2, 'gap_match': 27.2, 'scheme_fi...",9.87,4.86,1.51,0.78,0.45,1.41,2.3118,18.98,0.5339,9.90,2.2476,59.81,0.8564,52.33,-0.0258,-6.08,-0.6221,-78.96
3,918042953749565121,283,2025,25,6.8214,4.2143,2.1786,2.0357,0.3214,0.6786,"{'ast_per_game': 0.87, 'blk_per_game': 0.37, '...","{'dest_tier': 3, 'gap_match': 43.97, 'scheme_f...",6.10,3.71,0.87,0.56,0.37,1.08,0.7214,10.58,0.5043,11.97,1.3086,60.07,1.4757,72.49,-0.0486,-15.12,-0.4014,-59.15
4,8864662224880522052,135,2025,29,9.3125,7.7812,6.2188,1.5625,0.6562,1.1250,"{'ast_per_game': 0.15, 'blk_per_game': 0.04, '...","{'dest_tier': 2, 'gap_match': 12.0, 'scheme_fi...",1.22,0.55,0.15,0.09,0.04,0.22,8.0925,86.90,7.2312,92.93,6.0688,97.59,1.4725,94.24,0.6162,93.90,0.9050,80.44


## 2. §7a — Cohort splits (known categories, extends `compute_cohort_validation`'s slice definitions)

Position, archetype (M1, joined at *source* season — what kind of player this was heading into the portal), and tier direction (same derivation as the production pipeline's own `assign_competition_tiers`). Cheap, interpretable — do this first.

In [6]:
enriched = db.enrich_with_cohorts(engine, population_df)
residual_df = residual_df.merge(
    enriched[["player_id", "dest_school_id", "dest_season", "archetype_label", "tier_direction", "position"]],
    on=["player_id", "dest_school_id", "dest_season"],
    how="left",
)

overall = db.summarize_residuals(residual_df)
print("=== Overall ===")
overall

=== Overall ===


{'n': 2782.0,
 'pts_mean_residual': 1.216,
 'pts_median_residual': 0.903,
 'pts_rmse': 5.293,
 'pts_mae': 4.135,
 'reb_mean_residual': 0.442,
 'reb_median_residual': 0.107,
 'reb_rmse': 2.225,
 'reb_mae': 1.71,
 'ast_mean_residual': 1.293,
 'ast_median_residual': 1.1,
 'ast_rmse': 1.988,
 'ast_mae': 1.534,
 'stl_mean_residual': 0.363,
 'stl_median_residual': 0.22,
 'stl_rmse': 0.902,
 'stl_mae': 0.674,
 'blk_mean_residual': 0.033,
 'blk_median_residual': -0.048,
 'blk_rmse': 0.357,
 'blk_mae': 0.248,
 'tov_mean_residual': 0.404,
 'tov_median_residual': 0.148,
 'tov_rmse': 1.266,
 'tov_mae': 0.897}

In [7]:
by_position = db.summarize_residuals(residual_df, group_by="position")
pd.DataFrame(by_position).T

,n,pts_mean_residual,pts_median_residual,pts_rmse,pts_mae,reb_mean_residual,reb_median_residual,reb_rmse,reb_mae,ast_mean_residual,ast_median_residual,ast_rmse,ast_mae,stl_mean_residual,stl_median_residual,stl_rmse,stl_mae,blk_mean_residual,blk_median_residual,blk_rmse,blk_mae,tov_mean_residual,tov_median_residual,tov_rmse,tov_mae
C,670.0,0.876,0.771,4.629,3.673,1.432,1.340,2.907,2.297,1.972,1.680,2.529,2.030,1.129,1.067,1.407,1.160,0.275,0.193,0.580,0.426,-0.147,-0.126,0.686,0.543
PF,249.0,1.228,0.951,4.680,3.636,0.994,0.722,2.243,1.689,1.878,1.635,2.303,1.912,0.597,0.536,0.830,0.661,0.115,0.081,0.335,0.246,0.057,-0.020,0.807,0.620
PG,394.0,1.174,0.616,5.449,4.217,-0.571,-0.771,1.657,1.346,0.104,-0.039,1.211,0.953,-0.297,-0.409,0.517,0.452,-0.136,-0.160,0.199,0.171,2.016,2.017,2.491,2.109
SF,332.0,1.788,1.444,5.398,4.215,1.142,0.995,2.453,1.927,2.004,1.932,2.461,2.053,0.717,0.680,1.001,0.812,0.067,0.025,0.331,0.253,0.060,-0.062,0.844,0.663
SG,1137.0,1.261,0.957,5.686,4.464,-0.116,-0.376,1.824,1.431,0.968,0.833,1.569,1.210,-0.015,-0.108,0.538,0.427,-0.078,-0.114,0.209,0.169,0.347,0.203,1.070,0.815


In [8]:
by_archetype = db.summarize_residuals(residual_df, group_by="archetype_label")
pd.DataFrame(by_archetype).T

,n,pts_mean_residual,pts_median_residual,pts_rmse,pts_mae,reb_mean_residual,reb_median_residual,reb_rmse,reb_mae,ast_mean_residual,ast_median_residual,ast_rmse,ast_mae,stl_mean_residual,stl_median_residual,stl_rmse,stl_mae,blk_mean_residual,blk_median_residual,blk_rmse,blk_mae,tov_mean_residual,tov_median_residual,tov_rmse,tov_mae
Active Connector Forward,207.0,0.403,0.486,4.390,3.554,0.599,0.553,1.997,1.586,1.786,1.832,2.264,1.871,0.598,0.592,0.889,0.717,0.011,-0.029,0.247,0.187,-0.082,-0.281,0.896,0.715
High-Usage Frontcourt Creator,258.0,-0.175,-0.286,4.699,3.766,-0.154,-0.205,1.812,1.416,1.458,1.400,2.024,1.639,0.362,0.309,0.734,0.575,-0.054,-0.110,0.271,0.215,0.172,0.054,0.949,0.728
Interior Star Big,230.0,-0.404,-0.387,4.525,3.786,0.404,0.273,2.372,1.895,1.918,1.675,2.512,2.034,1.069,1.012,1.349,1.102,0.034,-0.005,0.378,0.291,-0.203,-0.365,0.798,0.664
Lead Scoring Playmaker,309.0,0.267,0.285,4.427,3.553,-0.960,-0.986,1.478,1.225,-0.072,-0.109,1.064,0.866,-0.367,-0.438,0.500,0.438,-0.165,-0.180,0.208,0.185,1.739,1.485,2.262,1.839
Post Scoring Big,218.0,-0.184,-0.925,4.186,3.498,0.874,0.750,2.382,1.887,2.016,1.801,2.520,2.049,1.201,1.189,1.473,1.241,0.192,0.135,0.557,0.412,-0.465,-0.544,0.702,0.602
Pressure Connector Guard,187.0,-0.003,-0.781,4.772,3.877,-0.446,-0.513,1.381,1.115,0.315,0.160,1.131,0.884,-0.202,-0.250,0.421,0.345,-0.112,-0.143,0.203,0.164,0.865,0.829,1.485,1.145
Skilled Stretch Forward,130.0,-0.534,-0.956,4.231,3.447,0.173,-0.058,1.883,1.466,1.597,1.484,2.092,1.672,0.482,0.383,0.797,0.588,0.001,-0.065,0.347,0.256,-0.356,-0.455,0.733,0.611
Two-Way Perimeter Guard,353.0,0.149,0.178,4.748,3.774,-0.953,-0.970,1.524,1.254,0.512,0.501,1.109,0.848,-0.340,-0.406,0.498,0.435,-0.153,-0.176,0.202,0.178,0.302,0.111,1.096,0.798
Two-Way Spacing Wing,261.0,-0.505,-0.912,4.520,3.702,-0.555,-0.568,1.380,1.117,0.794,0.748,1.253,0.979,-0.140,-0.211,0.418,0.354,-0.101,-0.133,0.193,0.162,-0.193,-0.322,0.806,0.670


In [9]:
by_tier_direction = db.summarize_residuals(residual_df, group_by="tier_direction")
pd.DataFrame(by_tier_direction).T

,n,pts_mean_residual,pts_median_residual,pts_rmse,pts_mae,reb_mean_residual,reb_median_residual,reb_rmse,reb_mae,ast_mean_residual,ast_median_residual,ast_rmse,ast_mae,stl_mean_residual,stl_median_residual,stl_rmse,stl_mae,blk_mean_residual,blk_median_residual,blk_rmse,blk_mae,tov_mean_residual,tov_median_residual,tov_rmse,tov_mae
down,776.0,4.093,3.501,6.555,5.112,1.781,1.609,2.846,2.218,1.935,1.798,2.421,2.001,0.651,0.524,1.073,0.810,0.193,0.092,0.436,0.291,0.705,0.434,1.344,0.943
same,1018.0,1.356,1.070,4.818,3.750,0.514,0.249,2.017,1.530,1.353,1.214,1.990,1.535,0.392,0.254,0.895,0.663,0.041,-0.040,0.348,0.237,0.437,0.180,1.259,0.876
up,988.0,-1.188,-1.363,4.606,3.764,-0.685,-0.814,1.841,1.496,0.727,0.561,1.562,1.167,0.106,-0.093,0.751,0.578,-0.101,-0.155,0.292,0.225,0.134,-0.126,1.210,0.884


## 3. §7b — Unsupervised clustering on residual vectors (exploratory)

Per-player residual vector across all 6 stats, scaled, k-means. Purpose: surface a bias mode nobody thought to slice by ahead of time — not a metric to optimize. Cluster count/interpretation need human review each run (same posture as M1/M2 clustering notebooks) — **inspect the silhouette/inertia plot and label the clusters yourself before trusting any conclusion.**

In [10]:
residual_cols = [f"residual_{stat}" for stat in db.BACKTEST_STATS]
cluster_input = residual_df[residual_cols].dropna()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(cluster_input.values)

# K-sweep — inspect inertia before picking a final K (same discipline as player_clustering.ipynb)
inertias = []
K_RANGE = range(2, 9)
for k in K_RANGE:
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

pd.Series(inertias, index=list(K_RANGE), name="inertia")

2    9870.256798
3    7950.539116
4    6589.046374
5    5879.486771
6    5299.945937
7    4952.050239
8    4620.534300
Name: inertia, dtype: float64

In [11]:
# Pick K from the elbow above, then inspect each cluster's mean residual vector —
# this is the "what bias pattern does this cluster represent" step, human-read.
K_FINAL = 4  # placeholder — set from the inertia plot above

km_final = KMeans(n_clusters=K_FINAL, n_init=10, random_state=42)
cluster_labels = km_final.fit_predict(X_scaled)

clustered = cluster_input.copy()
clustered["cluster"] = cluster_labels
cluster_profile = clustered.groupby("cluster")[residual_cols].mean()
cluster_profile["n"] = clustered.groupby("cluster").size()
cluster_profile

,residual_pts,residual_reb,residual_ast,residual_stl,residual_blk,residual_tov,n
cluster,,,,,,,
0,1.474870,0.836802,1.629433,0.618405,0.086496,0.025582,877
1,-3.754179,-1.618097,-0.002152,-0.234121,-0.208579,-0.394180,893
2,6.883322,3.823528,3.479895,1.497654,0.496146,0.793377,510
3,3.847494,-0.019554,0.785624,-0.175957,-0.100065,2.090139,502


## 4. Does the §7a cohort split already explain what §7b clustering finds?

Cross-tab the discovered clusters against position/archetype/tier_direction — if a cluster maps cleanly onto an existing cohort, clustering didn't find anything new. If a cluster cuts *across* existing cohorts, that's the useful finding — a bias mode nobody had a name for yet.

In [12]:
cohort_check = residual_df.loc[cluster_input.index, ["position", "archetype_label", "tier_direction"]].copy()
cohort_check["cluster"] = cluster_labels

print("Cluster x Position:")
display(pd.crosstab(cohort_check["cluster"], cohort_check["position"]))

print("\nCluster x Tier direction:")
display(pd.crosstab(cohort_check["cluster"], cohort_check["tier_direction"]))

print("\nCluster x Archetype (top 3 per cluster):")
for c in sorted(cohort_check["cluster"].unique()):
    top = cohort_check.loc[cohort_check["cluster"] == c, "archetype_label"].value_counts().head(3)
    print(f"  cluster {c}: {top.to_dict()}")

Cluster x Position:


position,C,PF,PG,SF,SG
cluster,,,,,
0,278,138,8,143,310
1,142,54,116,85,496
2,250,53,11,99,97
3,0,4,259,5,234



Cluster x Tier direction:


tier_direction,down,same,up
cluster,,,
0,265,352,260
1,108,291,494
2,271,175,64
3,132,200,170



Cluster x Archetype (top 3 per cluster):
  cluster 0: {'High-Usage Frontcourt Creator': 113, 'Active Connector Forward': 97, 'Interior Star Big': 96}
  cluster 1: {'Two-Way Perimeter Guard': 194, 'Two-Way Spacing Wing': 148, 'Lead Scoring Playmaker': 106}
  cluster 2: {'Post Scoring Big': 74, 'Interior Star Big': 63, 'Active Connector Forward': 37}
  cluster 3: {'Lead Scoring Playmaker': 192, 'Two-Way Perimeter Guard': 105, 'Pressure Connector Guard': 75}


## 5. Parse `explanation` deltas, dig into the two flagged findings

`explanation` (JSONB, already loaded onto `residual_df` via `load_projected_outcomes`) carries the real per-row delta breakdown `build_explanation_payload()` writes in production — flatten the fields needed below instead of re-deriving anything.

In [13]:
import json as _json


def _parse_explanation(x):
    if isinstance(x, str):
        try:
            return _json.loads(x)
        except (TypeError, ValueError):
            return {}
    return x or {}


_expl = residual_df["explanation"].map(_parse_explanation)
_EXPL_KEYS = [
    "role_usage_delta",
    "style_skill_fit_delta",
    "roster_context_delta",
    "competition_level_delta",
    "total_context_delta",
    "source_usage_rate",
    "dest_expected_usage",
    "dest_expected_minutes",
]
for _key in _EXPL_KEYS:
    residual_df[_key] = _expl.apply(lambda d, k=_key: d.get(k))

residual_df[_EXPL_KEYS].describe()

,role_usage_delta,style_skill_fit_delta,roster_context_delta,competition_level_delta,total_context_delta,source_usage_rate,dest_expected_usage,dest_expected_minutes
count,2782.000000,2782.000000,2782.000000,2782.000000,2782.000000,2782.000000,2782.000000,2782.000000
mean,0.234181,-0.004244,-0.349875,-0.049232,-0.169168,17.672682,19.507181,19.912132
std,0.551720,0.036776,0.140058,0.215235,0.599366,8.254416,3.274321,9.794107
min,-0.750000,-0.120200,-0.500000,-0.600000,-1.500000,0.000000,10.654500,0.870000
25%,-0.220325,-0.030350,-0.465750,-0.200000,-0.646025,14.900000,17.116225,11.830000
50%,0.415400,-0.002600,-0.378100,0.000000,-0.050300,19.000000,19.408250,23.335000
75%,0.750000,0.024675,-0.287600,0.120000,0.315725,23.100000,21.450650,27.300000
max,0.750000,0.104100,0.233500,0.360000,1.104700,37.300000,29.555200,34.570000


### 5a. Finding 1 — competition-tier bias

Real observed pattern: points residual trended positive on tier-up moves (model underprojects value gained moving to tougher competition) — check whether `competition_level_delta` (the model's own tier-adjustment term) tracks the actual residual, or whether the tier-adjustment magnitude is simply wrong-sized.

In [ ]:
tier_bias = residual_df.groupby("tier_direction").agg(
    n=("residual_pts", "size"),
    residual_pts_mean=("residual_pts", "mean"),
    residual_pts_median=("residual_pts", "median"),
    competition_level_delta_mean=("competition_level_delta", "mean"),
).round(3)
print(tier_bias)

_corr_tier = residual_df[["residual_pts", "competition_level_delta"]].dropna().corr().iloc[0, 1]
print(f"\ncorr(residual_pts, competition_level_delta) = {_corr_tier:.3f}")
if _corr_tier > 0.1:
    print("Positive corr: adjustment is right-signed but undersized -- rows where the model already")
    print("applied its biggest tier-down bump still carry the biggest leftover positive residual")
    print("(see tier-down row above: adj mean +0.131 vs real residual +4.09 -- ~30x too small).")
elif _corr_tier < -0.1:
    print("Negative corr: model's tier adjustment moves the wrong direction relative to the real residual (wrong-signed, not just wrong-sized).")
else:
    print("Near-zero corr: tier adjustment has no relationship to the real residual.")

### 5b. Finding 2 — turnover under-projection for guards

Real observed pattern: PGs assuming a bigger role at the destination school appear to turn the ball over more than projected. Bin by the model's own `dest_expected_usage` to see whether the miss scales with role size, and check both `dest_expected_usage` and `role_usage_delta` as candidate drivers.

In [15]:
pg_df = residual_df[residual_df["position"] == "PG"].copy()
pg_df["usage_bucket"] = pd.qcut(pg_df["dest_expected_usage"], q=4, duplicates="drop")

usage_bias = pg_df.groupby("usage_bucket", observed=True).agg(
    n=("residual_tov", "size"),
    residual_tov_mean=("residual_tov", "mean"),
    residual_tov_median=("residual_tov", "median"),
    dest_expected_usage_mean=("dest_expected_usage", "mean"),
).round(3)
print(usage_bias)

_corr_usage = pg_df[["residual_tov", "dest_expected_usage"]].dropna().corr().iloc[0, 1]
_corr_role = pg_df[["residual_tov", "role_usage_delta"]].dropna().corr().iloc[0, 1]
print(f"\ncorr(residual_tov, dest_expected_usage) = {_corr_usage:.3f}")
print(f"corr(residual_tov, role_usage_delta)     = {_corr_role:.3f}")
print("Positive corr on either = bigger destination role -> bigger real under-projection of turnovers (model doesn't scale TOV risk with role growth).")

                              n  residual_tov_mean  residual_tov_median  \
usage_bucket                                                              
(12.988000000000001, 19.39]  99              2.119                2.197   
(19.39, 21.677]              98              1.967                1.880   
(21.677, 23.223]             98              2.131                2.151   
(23.223, 29.555]             99              1.847                1.910   

                             dest_expected_usage_mean  
usage_bucket                                           
(12.988000000000001, 19.39]                    17.210  
(19.39, 21.677]                                20.643  
(21.677, 23.223]                               22.384  
(23.223, 29.555]                               25.615  

corr(residual_tov, dest_expected_usage) = -0.071
corr(residual_tov, role_usage_delta)     = -0.027
Positive corr on either = bigger destination role -> bigger real under-projection of turnovers (model doesn't sca